> 在线讲义：[第 10 章 · 转换与矩阵：Q、K、V 是怎么变出来的](https://wikieden.github.io/transformer-visual/lessons/10-matrix-transform.html) · 本书《从零读懂 Transformer》

# 第 10 章 · 转换与矩阵：Q、K、V 是怎么变出来的

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/10-matrix-transform.html) · 仓库里 `transformer-deep-dive/notes/lessons/10-matrix-transform.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 一组权重做点积 | 0.5×7 + 0×5 + 0.2×8 | `x @ W[:, 0]` | 5.1 |
| 向量乘矩阵 | $\mathbf x W$ = 和每一列各点积 | `x @ W` | [5.1, 3.2] |
| 形状 | 3 行 × 2 列：3 个进、2 个出 | `W.shape` | (3, 2) → 结果 (2,) |
| PyTorch 的线性层 | 同上 | `nn.Linear(3, 2, bias=False)` | [5.1, 3.2]，但它存的 weight 是 2 × 3 |
| 论文 $W^Q$、$W^K$、$W^V$ | 512 × 64 | `x @ w_q` | q、k、v 各 (64,)，三个不一样 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
from torch import nn
APPLE = torch.tensor([7.0, 5.0, 8.0])  # [甜, 大, 红]
W = torch.tensor([[0.5, 0.1], [0.0, 0.5], [0.2, 0.0]])

**函数 `by_columns`**：x 和 W 的每一列各做一次点积，排成新向量。

In [ ]:
def by_columns(x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """x 和 W 的每一列各做一次点积，排成新向量。"""
    return torch.stack([x @ w[:, j] for j in range(w.shape[1])])

**函数 `as_linear`**：nn.Linear(3, 2) 存的 weight 是 2 × 3（输出 × 输入），是讲义 W 的转置。

In [ ]:
def as_linear(w: torch.Tensor) -> nn.Linear:
    """nn.Linear(3, 2) 存的 weight 是 2 × 3（输出 × 输入），是讲义 W 的转置。"""
    layer = nn.Linear(w.shape[0], w.shape[1], bias=False)
    with torch.no_grad():
        layer.weight.copy_(w.T)
    return layer

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("一列一列点积:", [round(float(v), 2) for v in by_columns(APPLE, W)])
print(
    "x @ W       :",
    [round(float(v), 2) for v in APPLE @ W],
    "  形状",
    tuple(W.shape),
    "→",
    tuple((APPLE @ W).shape),
)
layer = as_linear(W)
print(
    "nn.Linear   :",
    [round(float(v), 2) for v in layer(APPLE).detach()],
    "  它的 weight 形状",
    tuple(layer.weight.shape),
)

In [ ]:
print("\n论文尺寸：同一个 x 乘三个不同的 512 × 64 矩阵")
torch.manual_seed(0)
x = torch.randn(512)
w_q, w_k, w_v = (torch.randn(512, 64) for _ in range(3))
q, k, v = x @ w_q, x @ w_k, x @ w_v
print("  q、k、v 形状:", tuple(q.shape), tuple(k.shape), tuple(v.shape))
print("  三个一样吗:", torch.allclose(q, k), torch.allclose(k, v))

## 3. 代码里才看得到的

1. **`nn.Linear` 存的是 $W$ 的转置**：讲义的 $W$ 是"进 × 出"（3 × 2），`nn.Linear(3, 2)` 的 `weight` 是"出 × 进"（2 × 3），算的是 `x @ weight.T`。第 22 章共用 embedding 表时，正是这个习惯让它不用转置。
2. **`with torch.no_grad()`**（no gradient：不记梯度）：手动往参数里写数时要包在里面，不然 PyTorch 会把"写入"也当成要求导的一步。

## 4. 自己改一改

- 香蕉 [8, 3, 0] 乘 `W`（题 2）：先手算，再跑 `torch.tensor([8.0, 3.0, 0.0]) @ W`。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一组权重做点积，得到一个新特征；几组权重竖着排，就是矩阵。
> 向量乘矩阵，竖着取列做点积；行数是原长度，列数是新长度。
> 同一个 x 乘三个矩阵，变出 q、k、v；矩阵里的数靠训练自己调。